In [1]:
from pyspark.sql import functions as F

# CONFIG
OUT_TABLE = "emodnet_chemistry_integrated"
KEYS = ["time", "latitude", "longitude"]

spark.conf.set("spark.sql.shuffle.partitions", "200")  # ajuste se precisar

def normalize_keys(df):
    return (df
        .withColumn("latitude",  F.round(F.col("latitude").cast("double"), 6))
        .withColumn("longitude", F.round(F.col("longitude").cast("double"), 6))
        .withColumn("time", F.col("time"))  # garante existência
    )

def prep(table, value_col, out_name):
    df = spark.table(table)
    df = normalize_keys(df)
    df = df.select(*KEYS, F.col(value_col).cast("double").alias(out_name))
    df = df.dropDuplicates(KEYS)
    return df

din = prep("emodnet_din_deepest", "DIN_deepest", "DIN")
do  = prep("emodnet_do_deepest", "DO_deepest", "DO")
pho = prep("emodnet_phosphate_deepest", "PHO_deepest", "PHO")
sil = prep("emodnet_silicate_deepest", "SIL_deepest", "SIL")
chl = prep("emodnet_chlorophyll_a_deepest", "CHL_a_deepest", "CHL_a")

# Repartition by join keys (ajuda no shuffle)
din = din.repartition(200, *KEYS)
do  = do.repartition(200, *KEYS)
pho = pho.repartition(200, *KEYS)
sil = sil.repartition(200, *KEYS)
chl = chl.repartition(200, *KEYS)

# JOIN correto (agora 1:1 por KEYS, sem explosão)
df = (din
    .join(do,  on=KEYS, how="inner")
    .join(pho, on=KEYS, how="inner")
    .join(sil, on=KEYS, how="inner")
    .join(chl, on=KEYS, how="inner")
    .withColumn("year", F.year("time"))
)

# Salvar (overwrite recomendado para rebuild)
(df.write
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .format("delta")
  .saveAsTable(OUT_TABLE)
)

print(f"✅ Tabela integrada criada: {OUT_TABLE}")

spark.table(OUT_TABLE).select("time","latitude","longitude","DIN","DO","PHO","SIL","CHL_a").show(10, truncate=False)


StatementMeta(, db229a0d-8e7b-4811-a80d-e7de9e320fca, 3, Finished, Available, Finished, False)

✅ Tabela integrada criada: emodnet_chemistry_integrated
+-------------------+--------+---------+---------+---------+-----------+---------+------------+
|time               |latitude|longitude|DIN      |DO       |PHO        |SIL      |CHL_a       |
+-------------------+--------+---------+---------+---------+-----------+---------+------------+
|2016-02-16 00:00:00|40.25   |3.625    |7.758482 |192.58109|0.39592835 |8.132908 |0.01172773  |
|2016-02-16 00:00:00|41.5    |3.625    |7.644447 |194.25696|0.40825734 |8.885521 |0.007181219 |
|2016-02-16 00:00:00|37.625  |2.375    |7.294586 |194.6232 |0.35825783 |9.094606 |0.0058132326|
|2016-02-16 00:00:00|39.375  |1.375    |7.8597555|201.1762 |0.37745875 |8.842921 |0.006234032 |
|2016-02-16 00:00:00|36.875  |1.75     |7.2737236|195.86177|0.35384774 |9.103194 |0.0053718886|
|2016-02-16 00:00:00|41.5    |2.625    |3.0254774|237.7715 |0.065389395|2.0992687|0.2477799   |
|2016-02-16 00:00:00|39.125  |4.5      |7.3885746|195.45479|0.3775349  |9.03858 